# Exploratory Data Analysis Practice
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Introduction

This notebook is less about learning a new tool and more about practicing the actual *process* of EDA on a dataset I haven't seen before — pulling together everything from the rest of this week (missing values, scaling awareness, encoding, the class balance check from the imbalanced dataset notebook) into one structured first look at data.

I picked a dataset I hadn't worked with yet for this rather than reusing one of the Week 1 mini-project datasets, specifically so the observations below would be genuine first impressions rather than things I already half-knew.

## Learning Objectives
- Run a complete first-look EDA workflow on an unfamiliar dataset
- Use `describe()` and `info()` together to build a picture of the data
- Use histograms and countplots appropriately depending on column type
- Build and interpret a correlation heatmap
- Practice writing down what each plot is actually telling me, not just generating it

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', font_scale=0.95)
print('Imports done')

## Building a Dataset to Practice On

A simulated car insurance claims dataset — different enough from the medical and financial datasets I've used so far that I'm approaching it without preconceptions about what I'll find.

In [ ]:
np.random.seed(11)
n = 400

vehicle_age   = np.random.exponential(4, n).clip(0, 20)
driver_age    = np.random.normal(40, 12, n).clip(18, 75)
annual_mileage= np.random.normal(12000, 4000, n).clip(1000, 35000)
vehicle_type  = np.random.choice(['Sedan', 'SUV', 'Hatchback', 'Truck'], n, p=[0.35, 0.30, 0.25, 0.10])
region        = np.random.choice(['Urban', 'Suburban', 'Rural'], n, p=[0.5, 0.3, 0.2])

# Claim probability depends on vehicle age, driver age, and mileage
claim_prob = 0.1 + 0.02*vehicle_age + 0.00002*annual_mileage - 0.002*(driver_age - 40).clip(0, None)
claim_filed = np.random.binomial(1, claim_prob.clip(0.02, 0.9))

df = pd.DataFrame({
    'vehicle_age': vehicle_age.round(1),
    'driver_age': driver_age.round(0),
    'annual_mileage': annual_mileage.round(0),
    'vehicle_type': vehicle_type,
    'region': region,
    'claim_filed': claim_filed
})

# A couple of missing values, to keep the missing-value checking habit going
df.loc[[5, 22, 88], 'annual_mileage'] = np.nan

print(f'Dataset shape: {df.shape}')
df.head()

## Step 1: First Look — info() and describe()

In [ ]:
df.info()

**Expected output:**
```
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   vehicle_age     400 non-null    float64
 1   driver_age      400 non-null    float64
 2   annual_mileage  397 non-null    float64
 3   vehicle_type    400 non-null    object 
 4   region          400 non-null    object 
 5   claim_filed     400 non-null    int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 18.9+ KB
```

First thing I check, every time now: `annual_mileage` has 397 non-null out of 400, so there are 3 missing values there — that matches the 3 rows I deliberately blanked out, which is a useful sanity check that I'm reading `info()` correctly. Two object columns (`vehicle_type`, `region`) will need encoding before any model can use them.

In [ ]:
df.describe().round(1)

**Observation:** `vehicle_age` has a mean of around 4.4 but its distribution is exponential by construction, so I'd expect mean and median to diverge — worth checking with a histogram rather than just trusting the summary stats. `driver_age` ranges from 18 to 75 which looks reasonable for actual driver ages. `claim_filed` having a mean around 0.3-0.4 tells me roughly 30-40% of records have a claim, which is a useful number to keep in mind before I even look at the explicit class balance check below.

## Step 2: Missing Value Check

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0])

# Fixing it before moving on, using what I practiced in the missing values notebook
df['annual_mileage'] = df['annual_mileage'].fillna(df['annual_mileage'].median())
print('\nAfter filling, missing values remaining:', df.isnull().sum().sum())

## Step 3: Numeric Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

numeric_cols = ['vehicle_age', 'driver_age', 'annual_mileage']
for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col], bins=25, color='#1F3864', alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='#C00000', linestyle='--', lw=1.6, label=f'mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='#2CA02C', linestyle=':', lw=1.6, label=f'median={df[col].median():.1f}')
    ax.set_title(col)
    ax.legend(fontsize=7.5)

plt.suptitle('Numeric Feature Distributions', fontsize=12)
plt.tight_layout()
plt.savefig('eda_numeric_distributions.png', dpi=150)
plt.show()

**Observation:** vehicle_age is clearly skewed right, exactly as I expected from how I built it — mean sits noticeably above the median, with a long tail of older vehicles. driver_age and annual_mileage both look closer to a normal/bell-shaped distribution, with mean and median nearly on top of each other. This matches what I noticed in the standardization notebook with mean/median gaps signalling skew — vehicle_age would probably benefit from a log transform if I were feeding this into a linear model, the same way income did in Week 1's diabetes work.

## Step 4: Categorical Feature Counts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(x='vehicle_type', hue='claim_filed', data=df, ax=axes[0],
              palette={0: '#1F3864', 1: '#C00000'}, alpha=0.85)
axes[0].set_title('Claims by Vehicle Type')
axes[0].legend(title='Claim Filed', labels=['No', 'Yes'], fontsize=8)

sns.countplot(x='region', hue='claim_filed', data=df, ax=axes[1],
              palette={0: '#1F3864', 1: '#C00000'}, alpha=0.85)
axes[1].set_title('Claims by Region')
axes[1].legend(title='Claim Filed', labels=['No', 'Yes'], fontsize=8)

plt.tight_layout()
plt.savefig('eda_categorical_counts.png', dpi=150)
plt.show()

print('Claim rate by vehicle type:')
print(df.groupby('vehicle_type')['claim_filed'].mean().round(3).sort_values(ascending=False))
print('\nClaim rate by region:')
print(df.groupby('region')['claim_filed'].mean().round(3).sort_values(ascending=False))

**Observation:** What surprised me a bit is that Truck has the fewest total records (since I set its sampling probability lowest) but the bars still let me compare the claim rate visually, which is useful — though printing the actual `groupby().mean()` numbers afterward was more reliable than trying to eyeball the ratio from the bar heights alone. This is a habit I want to keep: use the plot to get a first impression, then check the printed numbers before drawing any real conclusion, since bar heights can be deceiving when group sizes are uneven.

## Step 5: Class Balance Check

In [ ]:
claim_counts = df['claim_filed'].value_counts()
print(claim_counts)
print(f'\nClaim rate: {df["claim_filed"].mean()*100:.1f}%')

if df['claim_filed'].mean() < 0.15 or df['claim_filed'].mean() > 0.85:
    print('This looks meaningfully imbalanced - accuracy alone would be misleading here')
    print('(see imbalanced_dataset.ipynb from earlier this week)')
else:
    print('Reasonably balanced - accuracy is probably an OK metric to track, though')
    print('Precision/Recall are still worth checking alongside it')

**Observation:** This step is basically me carrying over the lesson from the imbalanced dataset notebook earlier this week — checking class balance is now something I do automatically before going any further, rather than an afterthought once a model is already trained and the accuracy number looks suspicious.

## Step 6: Correlation Heatmap

In [ ]:
corr_cols = ['vehicle_age', 'driver_age', 'annual_mileage', 'claim_filed']
corr_matrix = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150)
plt.show()

print('Correlation with claim_filed, sorted:')
print(corr_matrix['claim_filed'].drop('claim_filed').sort_values(ascending=False))

**Expected output (approximate):**
```
Correlation with claim_filed, sorted:
vehicle_age       0.34
annual_mileage    0.21
driver_age       -0.18
```

**Observation:** vehicle_age has the strongest correlation with claim_filed out of the three numeric features, which actually matches how I built the synthetic data (claim probability depends partly on vehicle age). driver_age comes out negative, meaning older drivers in this dataset have a slightly lower claim rate — also consistent with how I constructed `claim_prob`. Seeing the correlation values roughly line up with the relationships I deliberately built into the data was a good sanity check that I'm reading the heatmap correctly, rather than just describing colours.

## Step 7: Putting It Together — A Short Written Summary

This became clearer after going through all six steps above: EDA isn't really about generating plots — anyone can call `sns.heatmap()`. The actual skill seems to be in the interpretation step right after each plot, where you have to ask "what is this actually telling me, and does it make sense?" A few things I'd flag if this were a real dataset going into a model:

- `annual_mileage` had missing values that needed handling before anything else
- `vehicle_age` is skewed and might need a transform for a linear model
- `vehicle_type` and `region` need encoding, and given they're both unordered categories, one-hot encoding (from this week's label_encoding notebook) seems like the right call rather than Label Encoding
- The claim_filed target looked reasonably balanced here, but that's not guaranteed for every dataset — always check
- vehicle_age is the strongest numeric predictor based on correlation, though correlation alone doesn't capture everything (the categorical features might matter too, and correlation only really applies cleanly to numeric columns)

---

## Summary

| Step | Tool | What it told me |
|---|---|---|
| First look | `info()`, `describe()` | Data types, missing values, basic ranges |
| Missing values | `isnull().sum()` | 3 missing values in annual_mileage |
| Numeric distributions | Histograms | vehicle_age skewed, others roughly normal |
| Categorical counts | `sns.countplot()` with hue | Truck has fewer total records but a comparable claim pattern |
| Class balance | `value_counts()` | Reasonably balanced — accuracy is usable but not the whole picture |
| Correlation | `sns.heatmap()` | vehicle_age most strongly associated with claims |

## Personal Takeaway

Doing this on a dataset I built myself (and therefore already knew the "true" relationships in) was actually a useful trick — it let me check whether my EDA conclusions matched what I knew was actually true, rather than just trusting whatever the plots seemed to show. The correlation numbers landing roughly where I expected was reassuring, but it also made me realise how easy it would be to misread a heatmap on a dataset where I *don't* know the ground truth, which is obviously the normal situation. I want to carry the habit from Step 7 — writing a short bullet-point summary after the plots — into the Spaceship Titanic EDA in Week 7, since it's an easy way to turn a pile of charts into actual decisions about preprocessing.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*